# 16 · Subgraphs

**Goal:** by the end you can use a whole compiled graph as a single node inside a bigger graph, pass data in and out
of it, and draw the inside of it.

## The idea in plain English

As graphs grow, you want to build them from smaller parts, the way you build programs from functions. A **subgraph**
is a compiled graph used as a node inside another graph, the **parent graph**.

We'll build a small **arithmetic** graph (parse → route → compute) that you can run and test on its own. Then we use
it as **one node** inside a bigger **calculator app**: validate the input → arithmetic → format the answer.

How does data get in and out? If the parent and the subgraph **share** state keys (here `expression` and `result`),
LangGraph passes those keys through automatically. If they don't share keys, you call the subgraph from inside an
ordinary node and translate the keys yourself.

**Analogy:** a company with departments. The front desk (parent) takes a request, hands it to the arithmetic
department (subgraph), and gets back an answer. The front desk doesn't need to know how the department works inside.

## Picture

```mermaid
flowchart LR
    S([START]) --> validate --> arithmetic --> format --> E([END])
    subgraph arithmetic["arithmetic (a subgraph)"]
        direction LR
        parse -.-> add & subtract & multiply & divide
    end
```

## Part 1: The arithmetic subgraph

### Step 1: Its state and nodes

The arithmetic graph has its own state, `ArithState`. `a`, `op` and `b` are only needed inside it.

In [1]:
from typing import Literal, TypedDict

from langgraph.graph import StateGraph, START, END


class ArithState(TypedDict):
    expression: str  # shared with the parent
    a: float
    op: str
    b: float
    result: float  # shared with the parent

The nodes. Each operation is a separate node, as in lesson 05.

In [2]:
def parse(state: ArithState) -> dict:
    left, op, right = state["expression"].split()
    return {"a": float(left), "op": op, "b": float(right)}


def add(state: ArithState) -> dict:
    return {"result": state["a"] + state["b"]}


def subtract(state: ArithState) -> dict:
    return {"result": state["a"] - state["b"]}


def multiply(state: ArithState) -> dict:
    return {"result": state["a"] * state["b"]}


def divide(state: ArithState) -> dict:
    return {"result": state["a"] / state["b"]}

The router and the wiring. Each operation node leads to the subgraph's `END`, which is where control goes back to the parent.

In [3]:
OPERATIONS = {"+": "add", "-": "subtract", "*": "multiply", "/": "divide"}


def route(state: ArithState) -> Literal["add", "subtract", "multiply", "divide"]:
    return OPERATIONS[state["op"]]


arith_builder = StateGraph(ArithState)
arith_builder.add_node("parse", parse)
for name, fn in [("add", add), ("subtract", subtract), ("multiply", multiply), ("divide", divide)]:
    arith_builder.add_node(name, fn)
    arith_builder.add_edge(name, END)
arith_builder.add_edge(START, "parse")
arith_builder.add_conditional_edges("parse", route)

arithmetic = arith_builder.compile()
print(arithmetic.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse(parse)
	add(add)
	subtract(subtract)
	multiply(multiply)
	divide(divide)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse;
	parse -.-> add;
	parse -.-> divide;
	parse -.-> multiply;
	parse -.-> subtract;
	add --> __end__;
	divide --> __end__;
	multiply --> __end__;
	subtract --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Step 2: Test it on its own

A subgraph is a normal compiled graph, so you can invoke and test it before using it anywhere else.

In [4]:
print(arithmetic.invoke({"expression": "3 * 4"}))

{'expression': '3 * 4', 'a': 3.0, 'op': '*', 'b': 4.0, 'result': 12.0}


## Part 2: The parent graph

### Step 3: The parent's state and nodes

The calculator app only cares about the text, the result and the final answer. Its state has no `a`, `op` or `b`.
`validate` tidies the input (removes extra spaces) and checks it has three parts. `format` builds the answer.

In [5]:
class AppState(TypedDict):
    expression: str  # shared with the subgraph
    result: float  # shared with the subgraph
    answer: str


def validate(state: AppState) -> dict:
    parts = state["expression"].split()
    if len(parts) != 3:
        raise ValueError(f"Expected 'number operator number', got {state['expression']!r}")
    return {"expression": " ".join(parts)}  # tidy: exactly one space between parts


def format_answer(state: AppState) -> dict:
    return {"answer": f"{state['expression']} = {state['result']:g}"}

### Step 4: Add the subgraph as a node

`add_node("arithmetic", arithmetic)` takes the **compiled** subgraph where you'd normally pass a function. To the
parent it's just one node.

In [6]:
app_builder = StateGraph(AppState)
app_builder.add_node("validate", validate)
app_builder.add_node("arithmetic", arithmetic)  # a whole graph as one node
app_builder.add_node("format", format_answer)
app_builder.add_edge(START, "validate")
app_builder.add_edge("validate", "arithmetic")
app_builder.add_edge("arithmetic", "format")
app_builder.add_edge("format", END)

app = app_builder.compile()
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	validate(validate)
	arithmetic(arithmetic)
	format(format)
	__end__([<p>__end__</p>]):::last
	__start__ --> validate;
	arithmetic --> format;
	validate --> arithmetic;
	format --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Step 5: Run the parent

The messy input `"  3   *  4 "` is tidied by `validate`. Then the shared key `expression` flows **into** the
subgraph, and the shared key `result` flows back **out**.

Look at the final state: there's no `a`, `op` or `b`. Those keys exist only in the subgraph's state, so they stayed
inside it.

In [7]:
print(app.invoke({"expression": "  3   *  4 "}))

{'expression': '3 * 4', 'result': 12.0, 'answer': '3 * 4 = 12'}


### Step 6: See inside with `xray=True`

The normal diagram shows `arithmetic` as one box. `get_graph(xray=True)` opens it up and draws the subgraph's own
nodes inside a `subgraph arithmetic` block. (Mermaid writes the `:` in names such as `arithmetic:parse` as `\3a`.)

In [8]:
print(app.get_graph(xray=True).draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	validate(validate)
	format(format)
	__end__([<p>__end__</p>]):::last
	__start__ --> validate;
	arithmetic\3a__end__ --> format;
	validate --> arithmetic\3aparse;
	format --> __end__;
	subgraph arithmetic
	arithmetic\3aparse(parse)
	arithmetic\3aadd(add)
	arithmetic\3asubtract(subtract)
	arithmetic\3amultiply(multiply)
	arithmetic\3adivide(divide)
	arithmetic\3a__end__(<p>__end__</p>)
	arithmetic\3aparse -.-> arithmetic\3aadd;
	arithmetic\3aparse -.-> arithmetic\3adivide;
	arithmetic\3aparse -.-> arithmetic\3amultiply;
	arithmetic\3aparse -.-> arithmetic\3asubtract;
	arithmetic\3aadd --> arithmetic\3a__end__;
	arithmetic\3adivide --> arithmetic\3a__end__;
	arithmetic\3amultiply --> arithmetic\3a__end__;
	arithmetic\3asubtract --> arithmetic\3a__end__;
	end
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Step 7: Stream from inside the subgraph

From lesson 15: `stream` normally shows only the parent's steps, with the subgraph as one step. Passing
`subgraphs=True` also shows the subgraph's own steps. Each piece becomes `(namespace, update)`. The **namespace** is
`()` for the parent and names the subgraph for steps inside it.

In [9]:
for namespace, update in app.stream({"expression": "3 + 4"}, stream_mode="updates", subgraphs=True):
    where = "parent" if namespace == () else namespace[0].split(":")[0]
    print(f"{where:10} {update}")

parent     {'validate': {'expression': '3 + 4'}}
arithmetic {'parse': {'a': 3.0, 'op': '+', 'b': 4.0}}
arithmetic {'add': {'result': 7.0}}
parent     {'arithmetic': {'expression': '3 + 4', 'result': 7.0}}
parent     {'format': {'answer': '3 + 4 = 7'}}


## Part 3: When the schemas don't match

### Step 8: Call the subgraph from inside a node

Suppose a different app keeps its data under `question` and `reply`. It shares **no** keys with `ArithState`. Then
you can't add the subgraph as a node directly. Instead you write a normal node that calls `arithmetic.invoke(...)`,
**mapping** the keys on the way in (`question` → `expression`) and on the way out (`result` → `reply`).

In [10]:
class ChatState(TypedDict):
    question: str
    reply: str


def ask_arithmetic(state: ChatState) -> dict:
    sub_result = arithmetic.invoke({"expression": state["question"]})  # map keys in
    return {"reply": f"{state['question']} is {sub_result['result']:g}"}  # map keys out


chat_builder = StateGraph(ChatState)
chat_builder.add_node("ask_arithmetic", ask_arithmetic)
chat_builder.add_edge(START, "ask_arithmetic")
chat_builder.add_edge("ask_arithmetic", END)
chat = chat_builder.compile()

print(chat.invoke({"question": "6 * 7"}))

{'question': '6 * 7', 'reply': '6 * 7 is 42'}


**Which way should I use?**

| | Subgraph as a node | Call it inside a node |
|---|---|---|
| How | `add_node("name", compiled_subgraph)` | `def node(state): sub.invoke({...})` |
| Data in/out | Shared keys pass through automatically | You map the keys yourself |
| Use when | Parent and subgraph share keys | The schemas are different |
| `xray` and `subgraphs=True` streaming | Shows the inside | The subgraph is hidden in the node |

## What just happened

- **Steps 1–2** built and tested the `arithmetic` graph on its own: `3 * 4` gave `result: 12.0`.
- **Step 4** added the compiled `arithmetic` graph as a single node. The parent's diagram shows
  `validate --> arithmetic --> format`.
- **Step 5** ran the parent on `"  3   *  4 "`. It returned `expression: '3 * 4'`, `result: 12.0` and the answer, with
  no `a`, `op` or `b`: those stayed inside the subgraph.
- **Step 6** drew the inside with `xray=True`: a `subgraph arithmetic` block with its own `parse`, `add`, `subtract`,
  `multiply` and `divide`.
- **Step 7** streamed with `subgraphs=True`. The `parse` and `add` steps were labelled `arithmetic`; the rest were
  `parent`.
- **Step 8** used the subgraph from a parent with different keys, by calling it inside a node and mapping
  `question → expression` and `result → reply`.

## Common mistakes

1. **Adding a subgraph that shares no keys with the parent.** `chat_builder.add_node("arithmetic", arithmetic)` in
   `ChatState` runs the subgraph with nothing it can read:
   `KeyError: 'expression'`.
   Fix: share the keys, or call the subgraph inside a node and map them (Step 8).
2. **Passing the builder instead of the compiled graph.** `add_node("arithmetic", arith_builder)` fails:
   `TypeError: Expected a Runnable, callable or dict.Instead got an unsupported type: <class
   'langgraph.graph.state.StateGraph'>`.
   Fix: `add_node("arithmetic", arith_builder.compile())`.
3. **Expecting the subgraph's own keys in the parent's result.** `app.invoke(...)["op"]` fails with `KeyError: 'op'`.
   Only keys in the parent's schema come back.
   Fix: add the key to the parent's state if the parent needs it.

## Try it

1. Add `op: str` to `AppState` and rebuild the parent. Run `"3 * 4"` again. Does `op` appear in the result now? Why?
2. Write a parent node `times_ten` that runs **after** `arithmetic` and calls `arithmetic.invoke` a second time, with
   the expression `f"{state['result']:g} * 10"`, storing that result. For `"3 + 4"` you should get `70`.

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **Subgraph**: a compiled graph used as a node inside another graph.
- **Parent graph**: the graph that contains a subgraph.
- **Shared keys**: state keys that both the parent and the subgraph have. They pass in and out automatically.
- **`xray=True`**: `get_graph(xray=True)` draws the inside of subgraphs too.
- **Namespace**: in streaming with `subgraphs=True`, a label saying which graph a piece came from (`()` is the parent).